In [22]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from konlpy.tag import Komoran
from collections import Counter
# tqdm : 진행 상태를 로그로 표시하는 기능
from tqdm import tqdm

In [49]:
df = pd.read_csv('../data/ratings_train.txt', sep='\t')

df. dropna(inplace = True)

df.drop_duplicates('document', inplace = True)

df = df[:5000]

len(df)

5000

In [50]:
df['label'].value_counts()

label
0    2502
1    2498
Name: count, dtype: int64

In [51]:
komoran = Komoran()
# 모든 품사를 이용하니 학습의 능력이 떨어진다
# tokenized_sentence = [komoran.morphs(text) for text in df['document']]
# 품사를 필터링
def tokenize(text):
    allow_pos = ['NNP', 'NNG', 'VV','VA','MAG','SL']
    result = []
    for word, pos in komoran.pos(text):
        if pos in allow_pos :
            result.append(word)
        return result
tokenized_sentence = [tokenize(text) for text in df['document']]

In [52]:
# 단어 사전을 생성
# 패딩 토큰, 언노운 토큰 생성(초기 값)
vocab = {
    '<PAD>' : 0,
    '<UNK>' : 1
}
# tokenized_sentence에서 모든 토큰을 하나의 리스트로 생성
all_tokens = [token for tokens in tokenized_sentence for token in tokens]
# token들의 빈도수를 확인 -> min_count로 제한
token_counts = Counter(all_tokens)
token_counts

Counter({'정말': 90,
         '진짜': 74,
         '영화': 71,
         '보': 65,
         '평점': 55,
         '재밌': 53,
         '그냥': 50,
         '너무': 47,
         '이건': 40,
         '나': 36,
         '최고': 35,
         '좋': 35,
         '감독': 25,
         '어리': 25,
         '이': 25,
         '내용': 23,
         '솔직히': 22,
         '스토리': 21,
         '왜': 20,
         '재미있': 20,
         '마지막': 19,
         '완전': 18,
         '감동': 18,
         '말': 18,
         '오': 17,
         '이렇': 17,
         '기대': 17,
         '재미': 17,
         '이영화': 16,
         '지금': 16,
         '쓰레기': 16,
         '시간': 15,
         '처음': 14,
         '재미없': 13,
         '배우': 13,
         '최악': 13,
         '짱': 13,
         '잘': 13,
         '가슴': 13,
         '오랜만': 12,
         '일본': 12,
         '굿': 12,
         '남자': 12,
         '아직': 11,
         '소재': 11,
         '제목': 11,
         '원작': 10,
         '참': 10,
         '사랑': 10,
         '별로': 10,
         '여자': 10,
         '한국': 9,
         '좀': 9,

In [53]:
# 단어의 빈도수가 3이상인 토큰들만 이용하여 단어 사전에 넣어준다
for token, count in token_counts.items():
    if count >= 3:
        vocab[token] = len(vocab)

In [54]:
vocab

{'<PAD>': 0,
 '<UNK>': 1,
 '원작': 2,
 '액션': 3,
 '왜': 4,
 '볼': 5,
 '울': 6,
 '참': 7,
 '이건': 8,
 '보': 9,
 '재미없': 10,
 '다': 11,
 '엄': 12,
 '졸': 13,
 '재밌': 14,
 '아직': 15,
 '포스터': 16,
 '오': 17,
 '정말': 18,
 '평점': 19,
 '나': 20,
 '영화': 21,
 '이렇': 22,
 '난': 23,
 '재미있': 24,
 '최고': 25,
 '너무': 26,
 '사랑': 27,
 '많': 28,
 '예전': 29,
 '내용': 30,
 '한국': 31,
 '감독': 32,
 '진정': 33,
 '별루': 34,
 '성룡': 35,
 '완전': 36,
 '인상': 37,
 '진짜': 38,
 '설정': 39,
 '무섭': 40,
 '킬링타임': 41,
 '음악': 42,
 '왕': 43,
 '솔직히': 44,
 '대박': 45,
 '시청률': 46,
 '소재': 47,
 '그냥': 48,
 '별점': 49,
 '별로': 50,
 '아니': 51,
 '아주': 52,
 '당시': 53,
 '케이블': 54,
 '좋': 55,
 '작가': 56,
 '명작': 57,
 '배우': 58,
 '어린이': 59,
 '오랜만': 60,
 '어리': 61,
 '알바': 62,
 '기존': 63,
 '이영화': 64,
 '이': 65,
 '넘': 66,
 'TV': 67,
 '새벽': 68,
 '감동': 69,
 '너무너무': 70,
 '진심': 71,
 '더럽': 72,
 '현실': 73,
 '오늘': 74,
 '코믹': 75,
 '일단': 76,
 '어떻': 77,
 '좀': 78,
 '일본': 79,
 '세계': 80,
 '엄마': 81,
 '그저': 82,
 '드라마': 83,
 '연기': 84,
 '중간': 85,
 'OOO': 86,
 '사람': 87,
 '여주인공': 88,
 '스토리': 89,
 '나이': 90,
 '

In [55]:
# vocab을 이용한 토큰화 된 데이터의 인코딩과 Dataset을 결합
# dict.get() -> 특정 키를 입력하면 해당 키의 값을 되돌려주는 함수
# (두번째 인자값을 이용하여 첫번째 인자의 키 값이 존재하지 않을 때 디폴트 값을 설정)
vocab.get('마케팅', vocab['<UNK>'])

1

In [56]:
class RNNDataset(Dataset):
    # 생성자, 깊이 출력함수, 특정 위치의 데이터 출력함수
    def __init__(self, tokenized_texts, labels, vocab):
        # tokenized_text : 토큰화된 문서들(독립 변수)
        # lables : 정답 데이터 (종속 변수)
        # vocab : 단어 사전
        self.labels = labels.values
        self.data = [
            [
                vocab.get(token, vocab['<UNK>']) for token in tokens
            ]
            for tokens in tokenized_texts
        ]
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        # getitem 함수의 역할 : DataLoader가 데이터를 불러오는 함수
        return torch.tensor(self.data[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)
    

In [57]:
# 후 처리 가공 함수(DataLoader가 배치 사이즈만큼 Dataset을 불러오고 불러온 데이터를 후 처리 가공)
def collate_fn(batch):
    # 배치 단위로 들어온 데이터를 최대 길이의 data에 맞게 패딩 토큰을 채워준다.
    # 배치 -> [ (data, label), (data, label), ... ]
    text_list = [item[0] for item in batch]
    label_list = [item[1] for item in batch]

    # text_list에 있는 인코딩된 데이터에서 최대 길이만큼 나머지 데이터에 패딩 토큰을 채워준다
    padded_texts = pad_sequence(text_list, batch_first=True, padding_value=vocab['<PAD>'])
    labels = torch.tensor(label_list, dtype = torch.long)

    return padded_texts, labels

In [58]:
# Datset 생성
dataset = RNNDataset(tokenized_sentence, df['label'], vocab)
# train의 길이와 text의 길이를 설정
train_size = int(len(dataset) * 0.8) # int() 사용하는 이유는?
test_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, test_size])

In [59]:
print(len(train_dataset), len(val_dataset))

4000 1000


In [60]:
train_loader = DataLoader(train_dataset, batch_size = 64, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)

In [61]:
class RNNCLF(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_size, num_classes):
        # vocab_size : 임베딩 함수 입력 차원의 수
        # emb_dim : 임베딩 함수 출력 차원의 수
        # hidden_size : RNN 은닉층의 출력 차원의 수
        # num_classes : 선형 모델의 출력 차원의 수(분류 개수)
        super().__init__()
        # 입력되는 데이터는 인코딩 된 데이터 (2,3,4) -> 벡터화 작업 ( nn.Enbedding(), Word2Vec, FastText, Doc2Vec)
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=vocab['<PAD>'])
        # RNN 모델
        self.rnn = nn.RNN(emb_dim, hidden_size, batch_first=True)
        # 선형 모델
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x : DataLoader의 독립 변수 값(토큰화 데이터)
        embedding = self.emb(x) # [batch_size, seq_len, emb_dim]

        # rnn_out -> [batch_size, seq_len, hidden_size] (모든 시점의 출력)
        # hidden -> [1, seq_len, hidden_size]   (제일 마지막 시점의 은닉 상태)
        rnn_out, hidden = self.rnn(embedding)

        # 선형 모델에 데이터를 대입하기 위해서 hidden의 배치층을 제거
        last_hidden = hidden.squeeze(0) # [seq_len, hidden]

        return self.fc(last_hidden)

In [62]:
# 모델 생성
model = RNNCLF(len(vocab), emb_dim=64, hidden_size=128, num_classes=2)
# 손실 함수
criterion = nn.CrossEntropyLoss()
# 옵티마이저 생성
optimizer = optim.Adam(model.parameters(), lr = 0.001)

In [63]:
epochs = 50

for epoch in range(epochs):
    model.train()
    train_loss = 0
    correct_train = 0
    total_train = 0

    # tqdm() -> desc는 로그 출력 값
    for inputs, labels in tqdm(train_loader, desc = f'Epoch {epoch+1} / {epochs} Train'):
        optimizer.zero_grad()
        output = model(inputs)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        pred = torch.argmax(output, dim=1)
        correct_train += (pred == labels).sum().item()
        total_train += labels.size(0)
    
    train_acc = (correct_train / total_train) * 100
    avg_train_loss = train_loss / len(train_loader)

    # 검증 구간
    model.eval()
    val_loss = 0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            output = model(inputs)
            loss = criterion(output, labels)

            val_loss += loss.item()
            pred = torch.argmax(output, dim=1)
            correct_val += (pred == labels).sum().item()
            total_val += labels.size(0)
    val_acc = (correct_val / total_val) * 100
    avg_val_loss = val_loss / len(val_loader)
    if (epoch+1) % 10 == 0:
        print(f'RNN 에폭 : Train Loss : {round(avg_train_loss, 4)} Train Acc : {train_acc}')
        print(f'RNN 에폭 : Vali Loss : {round(avg_val_loss, 4)} Vali Acc : {val_acc}')

Epoch 10 / 50 Train: 100%|██████████| 63/63 [00:00<00:00, 145.00it/s]


RNN 에폭 : Train Loss : 0.6358 Train Acc : 58.675
RNN 에폭 : Vali Loss : 0.6459 Vali Acc : 57.599999999999994


Epoch 20 / 50 Train: 100%|██████████| 63/63 [00:00<00:00, 107.08it/s]


RNN 에폭 : Train Loss : 0.6226 Train Acc : 59.775
RNN 에폭 : Vali Loss : 0.6479 Vali Acc : 61.5


Epoch 30 / 50 Train: 100%|██████████| 63/63 [00:00<00:00, 168.27it/s]


RNN 에폭 : Train Loss : 0.6167 Train Acc : 59.724999999999994
RNN 에폭 : Vali Loss : 0.6495 Vali Acc : 62.1


Epoch 40 / 50 Train: 100%|██████████| 63/63 [00:00<00:00, 132.99it/s]


RNN 에폭 : Train Loss : 0.6157 Train Acc : 59.425
RNN 에폭 : Vali Loss : 0.6596 Vali Acc : 59.199999999999996


Epoch 50 / 50 Train: 100%|██████████| 63/63 [00:00<00:00, 149.11it/s]

RNN 에폭 : Train Loss : 0.6146 Train Acc : 59.275
RNN 에폭 : Vali Loss : 0.6647 Vali Acc : 58.4
